# ICMI Corpus — Building the Corpus-Wide Dataset
### Notebook 2 of 3: Consolidation and Export

This notebook reads all GAT-2 transcripts from two source formats:
1. **PDF transcripts** — publicly released corpus data (13 files)
2. **EXMARaLDA CSV exports** — project-internal working data

Both are converted into a uniform 14-column DataFrame and combined
into `csv_export/korpus_gesamt.csv` — the sole input for Notebook 3.

---


## Part 1 — Setup

Before Python can read files, it needs the right tools. In Python these tools are called **libraries** — ready-made code collections written by other developers.

We need:
- **`pdfplumber`** — reads text from PDF files
- **`pandas`** — creates and analyses data tables (like Excel, but for Python)
- **`re`** — searches for patterns in text (regular expressions)
- **`pathlib`** and **`glob`** — help with handling file paths

In [23]:
import subprocess
subprocess.run(["pip", "install", "pdfplumber"], capture_output=True)

import re
import glob
import pandas as pd
import pdfplumber
from pathlib import Path
from collections import Counter

print("All libraries loaded.")


All libraries loaded.


---
## Part 2 — Reading Files

All transcript files have the `.pdf` extension, but come in two variants:

1. **Real PDF files** — text is embedded in a complex format, recognisable by the file starting internally with the characters `%PDF`.
2. **Plain text files with a `.pdf` extension** — pure text, only the extension is misleading.

The following function detects both variants automatically.

In [24]:
def rohtext_aus_datei(dateipfad):
    """
    Reads raw text from a transcript file.
    Automatically detects whether it is a real PDF or a plain text file.
    """
    with open(dateipfad, "rb") as f:
        start = f.read(4)
    if start == b"%PDF":
        # Real PDF: pdfplumber extracts text page by page
        seiten = []
        with pdfplumber.open(dateipfad) as pdf:
            for seite in pdf.pages:
                text = seite.extract_text()
                if text:
                    seiten.append(text)
        return "\n".join(seiten)
    else:
        # Plain text file: read directly (encoding: UTF-8)
        return open(dateipfad, encoding="utf-8", errors="replace").read()

# Find all transcript files (filename starts with year)
alle_dateien = sorted(glob.glob("20*.pdf"))
print(f"{len(alle_dateien)} transcript files found:")
for d in alle_dateien:
    with open(d, "rb") as f:
        ist_pdf = f.read(4) == b"%PDF"
    print(f"  {'PDF    ' if ist_pdf else 'Text   '}  {d}")

13 transcript files found:
  PDF      2012_Uberaba_Alemaes_Sueca1.pdf
  PDF      2013_Muenster_Alemaes1-Parte1.pdf
  PDF      2013_Muenster_Alemaes1-Parte2.pdf
  PDF      2014_BeloHorizonte_Alemao_Brasileiros_AulaPreposicao01_parte2.pdf
  PDF      2014_BeloHorizonte_Alemao_Brasileiros_AulaPreposicao01_parte3.pdf
  PDF      2014_Muenster_Alemaes2-Parte1.pdf
  PDF      2014_Muenster_Alemaes2-Parte2.pdf
  PDF      2014_Muenster_Brasileiros2-Parte1.pdf
  PDF      2014_Muenster_Brasileiros2-Parte2.pdf
  PDF      2014_Muenster_Brasileiros2-Parte3.pdf
  PDF      2015_BeloHorizonte_Alemas_Brasileiras_Heimat1.pdf
  PDF      2015_BeloHorizonte_Brasileiros_Assembleia1.pdf
  PDF      2016_BeloHorizonte_Brasileiros_Assembleia2.pdf


---
## Part 2b — Reading EXMARaLDA CSV Exports

In addition to the GAT-2 text transcripts, some recordings are also available as **EXMARaLDA exports** — semi-automatically generated CSV files with precise timestamps from the EXMARaLDA Partitur-Editor.

**Format (no column header, semicolon- or tab-separated):**

```
Tier-Name  |  t  |  SPK_id  |  Text  |  Start_sec  |  End_sec  |  v  |  TIE_id
AD8 [v]    ;  t  ;  SPK4   ;  ãh,   ;  0.0        ;  0.713   ;  v  ;  TIE4
```

The **tier name** (`AD8 [v]`) encodes speaker ID and channel.
The `T [v]` tier contains pauses (e.g. `(1.2)`).

Since EXMARaLDA exports do not have their own speaker table,
speaker metadata can be entered in the `EXMARALDA_META` dictionary below.
Missing entries will be set to `unknown`.

In [25]:
# ── Speaker metadata for EXMARaLDA files ──────────────────────────────────
# Enter speaker metadata for each CSV file here.
# Format: {filename_without_extension: {speaker_id: {sex, l1, l2, occupation, age, education}}}
# Missing entries or missing fields will be set to 'unknown' automatically.
EXMARALDA_META = {
    "assembleia1": {
        "MGP": {"sex": "f", "l1": "por", "l2": "deu", "occupation": "prof"},
        "T":   {"sex": "unknown", "l1": "unknown", "l2": "unknown", "occupation": "unknown"},
    },
    "assembleia2": {
        "MGP": {"sex": "f", "l1": "por", "l2": "deu", "occupation": "prof"},
        "AD8": {"sex": "unknown", "l1": "deu", "l2": "por", "occupation": "stud"},
        "T":   {"sex": "unknown", "l1": "unknown", "l2": "unknown", "occupation": "unknown"},
    },
    # add further files here ...
}
print("Metadata dictionary loaded.")


Metadata dictionary loaded.


In [26]:
def sek_zu_zeitstr(sek):
    """
    Converts a seconds value (float) to MM:SS.s format
    — the same format used in the GAT-2 text transcripts.
    Example: 74.3 → '01:14.3'
    """
    if sek is None:
        return None
    m = int(sek) // 60
    s = sek - m * 60
    return f"{m:02d}:{s:04.1f}"

def exmaralda_zu_dataframe(csv_pfad, sprecher_meta=None):
    """
    Reads an EXMARaLDA CSV export and returns a pandas DataFrame
    with the same 14 columns as datei_zu_dataframe().
    csv_pfad      : path to the CSV file
    sprecher_meta : optional dict {speaker_id: {sex, l1, l2, occupation, age, education}}
    """
    # 1. Auto-detect separator
    with open(csv_pfad, encoding="utf-8", errors="replace") as f:
        erste_zeile = f.readline()
    sep = ";" if erste_zeile.count(";") >= erste_zeile.count("\t") else "\t"

    # 2. Read CSV (no header)
    spaltennamen = ["tier", "typ", "spk_id", "text",
                    "start_sek", "end_sek", "kanal_raw", "tie_id"]
    df_raw = pd.read_csv(
        csv_pfad, sep=sep, header=None, names=spaltennamen,
        encoding="utf-8", on_bad_lines="skip", engine="python"
    )
    df_raw = df_raw.dropna(subset=["tier", "start_sek", "end_sek"])
    df_raw["start_sek"] = pd.to_numeric(df_raw["start_sek"], errors="coerce")
    df_raw["end_sek"]   = pd.to_numeric(df_raw["end_sek"],   errors="coerce")
    df_raw = df_raw.dropna(subset=["start_sek", "end_sek"])

    # Empty after cleanup → skip this file
    if len(df_raw) == 0:
        return None
    df_raw = df_raw.reset_index(drop=True)

    # 3. Tier name → speaker + channel
    def parse_tier(tier_str):
        m = re.match(r"^(.+?)\s*\[(v|nv)\]", str(tier_str).strip())
        if m:
            return m.group(1).strip(), m.group(2)
        return str(tier_str).strip(), "v"
    tiers = df_raw["tier"].map(parse_tier)
    df_raw["sprecher"] = [t[0] for t in tiers]
    df_raw["kanal"]    = [t[1] for t in tiers]

    # 4. Sort chronologically
    df_raw = df_raw.sort_values("start_sek").reset_index(drop=True)

    # 5. Reconstruct turns:
    #    New turn when there is a gap of > 0.05 sec after the current window.
    turn_ids = []
    aktueller_turn = 1
    max_ende = df_raw.iloc[0]["end_sek"]
    for _, row in df_raw.iterrows():
        if row["start_sek"] > max_ende + 0.05:
            aktueller_turn += 1
        turn_ids.append(aktueller_turn)
        max_ende = max(max_ende, row["end_sek"])
    df_raw["turn_id"] = turn_ids

    # 6. Segment numbers (TIE_id as reference)
    df_raw["segmente"] = df_raw["tie_id"].apply(lambda x: [str(x)])

    # 7. Time strings
    df_raw["segment_start"] = df_raw["start_sek"].apply(sek_zu_zeitstr)
    df_raw["segment_ende"]  = df_raw["end_sek"].apply(sek_zu_zeitstr)

    # 8. Result DataFrame
    ergebnis = df_raw[["turn_id", "segment_start", "segment_ende", "segmente",
                        "sprecher", "kanal", "text", "tie_id"]].copy()
    ergebnis["text"]  = ergebnis["text"].fillna("").astype(str)
    ergebnis["datei"] = Path(csv_pfad).stem
    ergebnis = ergebnis.drop(columns=["tie_id"])

    # 9. Add speaker metadata
    standardfelder = ["sex", "l1", "l2", "occupation", "age", "education"]
    if sprecher_meta:
        meta = pd.DataFrame(sprecher_meta).T.reset_index().rename(
            columns={"index": "sprecher"}
        )
        ergebnis = ergebnis.merge(meta, on="sprecher", how="left")
    for feld in standardfelder:
        if feld not in ergebnis.columns:
            ergebnis[feld] = "unknown"
        else:
            ergebnis[feld] = ergebnis[feld].fillna("unknown")

    # 10. Uniform column order
    kernspalten = ["turn_id", "segment_start", "segment_ende", "segmente",
                   "sprecher", "kanal", "text", "datei",
                   "sex", "l1", "l2", "occupation", "age", "education"]
    ergebnis = ergebnis[[k for k in kernspalten if k in ergebnis.columns]]

    return ergebnis

print("Functions sek_zu_zeitstr() and exmaralda_zu_dataframe() defined.")

Functions sek_zu_zeitstr() and exmaralda_zu_dataframe() defined.


In [27]:
# Find EXMARaLDA CSV exports
# We look for CSVs that do NOT start with 'csv_export'
# (i.e. not the notebook's own output files).
exmaralda_dateien = [
    p for p in sorted(glob.glob("*.csv"))
    if not Path(p).stem.endswith("_auswertung")
    and not Path(p).stem.endswith("_gesamt")
]
print(f"{len(exmaralda_dateien)} EXMARaLDA CSV file(s) found:")
for d in exmaralda_dateien:
    meta_vorhanden = Path(d).stem in EXMARALDA_META
    print(f"  {'[Meta]' if meta_vorhanden else '[no meta]'}  {d}")

# Duplicate check: same files with different names?
from itertools import combinations
for a, b in combinations(exmaralda_dateien, 2):
    pa, pb = Path(a).stem.replace('_',''), Path(b).stem.replace('_','')
    if pa == pb or pa in pb or pb in pa:
        print(f"  ⚠ Possible duplicates: {a}  ↔  {b}  (please verify!)")

5 EXMARaLDA CSV file(s) found:
  [Meta]  assembleia1.csv
  [Meta]  assembleia2.csv
  [no meta]  assembleia_2.csv
  [no meta]  icmi_speaker_metadata.csv
  [no meta]  modalpartikeln.csv
  ⚠ Possible duplicates: assembleia2.csv  ↔  assembleia_2.csv  (please verify!)


---
## Part 3 — Understanding the Structure of a GAT-2 File

Each transcription file is divided into three sections:

```
HEADER          → Metaproject name, location, date, participants ...
SPEAKERTABLE    → Speaker IDs with L1, L2, gender, occupation ...
TURNS           → The actual conversation data [1] to [N]
```


Within the turns, there are four line types:

| Type | Example |
|---|---|
| Turn number | `[7]` |
| Timestamp | `18 [00:30.0] 19 [00:31.0]` |
| Verbal utterance | `A1 [v] ich verstehe nicht VIEL` |
| Pause / Non-verbal | `[nv] (1.2)` |

Important: Speaker IDs can have various formats — `A1`, `D2m`, `AD1`, `MGP` — depending on the project's transcription convention.

In [28]:
def kategorisiere(zeile):
    """
    Ordnet eine Zeile einem von sechs Typen zu.
    Grundlage sind Muster (reguläre Ausdrücke) am Zeilenanfang.
    """
    z = zeile.strip()
    if z == "":
        return "leer"
    # Turn-Nummer: nur [Zahl] in der Zeile
    if re.match(r"^\[\d+\]$", z):
        return "turn_nummer"
    # Zeitstempel: beginnt mit Zahl oder '..' gefolgt von Zahl + [Zeit]
    if re.match(r"^(\.\. )?\d+\s*\[", z):
        return "zeitstempel"
    # Verbale Äußerung: Sprecher-ID + [v]
    # Sprecher-IDs können sein: A1, D2m, AD1, MGP, T, Gruppe, ...
    if re.match(r"^[A-Za-z][A-Za-z0-9]*(?:\s+\S+)?\s+\[v\]", z):
        return "sprecher_verbal"
    # Non-verbale Äußerung eines Sprechers
    if re.match(r"^[A-Za-z][A-Za-z0-9]*(?:\s+\S+)?\s+\[nv\]", z):
        return "sprecher_nonverbal"
    # Pause / non-verbales Ereignis ohne Sprecher
    if re.match(r"^\[nv\]", z):
        return "nonverbal_solo"
    return "sonstiges"


In [29]:
# Test: Zeilentypen in einer Beispieldatei zählen
beispiel_datei = alle_dateien[0]
rohtext = rohtext_aus_datei(beispiel_datei)
zeilen  = rohtext.splitlines()

kategorien = Counter(kategorisiere(z) for z in zeilen)
print(f"Zeilentypen in: {Path(beispiel_datei).name}\n")
for kat, anzahl in kategorien.most_common():
    print(f"  {kat:22s}  {anzahl:5d}")


Zeilentypen in: 2012_Uberaba_Alemaes_Sueca1.pdf

  sonstiges                2959
  sprecher_verbal          2924
  turn_nummer              1618
  zeitstempel              1545


---
## Part 4 — Reading the Three Sections

We write a separate function for each section.

### 4.1 Header — Metadaten

In [30]:
def parse_header(header_zeilen):
    """
  Reads the metadata header as a dict: {field_name: value}.
Multi-line values (e.g., long project titles) are merged.
    """
    header = {}
    schluessel = None
    puffer = []

    for z in header_zeilen:
        z = z.strip()
        if not z:
            continue
        m = re.match(r"^([^:]+):\s*(.*)$", z)
        if m:
            if schluessel:
                header[schluessel] = " ".join(puffer).strip()
            schluessel = m.group(1).strip()
            puffer = [m.group(2).strip()]
        else:
            if schluessel:
                puffer.append(z)

    if schluessel:
        header[schluessel] = " ".join(puffer).strip()
    return header


### 4.2 Extracting Occupations from the Header

Some files list occupations in the format `Professoras (A2, B1) e Estudante (A1)`. We normalize these to uniform abbreviations: `prof` and `stud`.


In [31]:
def parse_berufe(header):
    """
    Extrahiert Sprecher -> Beruf aus der Profissão-Zeile im Header.
    """
    normalisierung = {
        "professoras": "prof", "professores": "prof",
        "professor":   "prof", "professora":  "prof",
        "estudante":   "stud", "estudantes":  "stud",
        "student":     "stud", "studentin":   "stud",
        "pastor":      "past", "pastora":     "past",
    }
    berufsinfo = header.get("Profissão dos participantes", "")
    sprecher_beruf = {}
    for treffer in re.finditer(r"(\w+)\s*\(([^)]+)\)", berufsinfo):
        beruf = normalisierung.get(treffer.group(1).lower(), treffer.group(1).lower())
        for spk in treffer.group(2).split(","):
            sprecher_beruf[spk.strip().split(":")[0].strip()] = beruf
    return sprecher_beruf


## 4.3 Speakertable — Speaker Metadata

The speaker table has different formats in different files:
- Standard: `A1` alone on a line
- With location: `D2m Warschau`
- Letters only: `MGP`, `T`

We recognize a speaker ID as a line that is short, contains no colon, and is not called `Speakertable`.


In [32]:
def parse_speakertable(speakertable_zeilen, sprecher_beruf):
    """
    Reads the speaker table.
Returns a dict: {speaker_ID: {sex, l1, l2, occupation, age, education}}

Field names are standardized (English).
Fields not in the standard set are ignored.
Missing values are entered as "unknown".
    """
    # Mapping: Original-Feldname -> einheitlicher englischer Name
    feld_mapping = {
        "sex":                    "sex",
        "l1":                     "l1",
        "l2":                     "l2",
        "languages_used":         None,           # wird nicht behalten
        "comment":                None,
        "user_defined_attributes": None,
        "profissão":              "occupation",
        "beruf":                  "occupation",
        "idade":                  "age",
        "escolaridade":           "education",
    }

    sprecher = {}
    aktueller = None

    for z in speakertable_zeilen:
        z_orig = z.strip()
        if not z_orig or z_orig.lower() == "speakertable":
            continue

        # Speaker ID: short line without colon, starts with capital letter
        if (len(z_orig) < 40 and ":" not in z_orig
                and re.match(r"^[A-Z]", z_orig)):
            aktueller = z_orig
            sprecher[z_orig] = {}
        elif aktueller:
            m = re.match(r"^([\w\s\u00c0-\u024f]+):\s*(.*)$", z_orig)
            if m:
                k_orig = m.group(1).strip().lower().replace(" ", "_")
                k_neu  = feld_mapping.get(k_orig)
                if k_neu is not None:         # keep only known fields
                    sprecher[aktueller][k_neu] = m.group(2).strip() or "unknown"

    # Supplement occupation info from header (only overwrites if not yet set)
    for spk in sprecher:
        if "occupation" not in sprecher[spk]:
            sprecher[spk]["occupation"] = sprecher_beruf.get(spk, "unknown")

    # Fill missing standard fields with "unknown"
    standardfelder = ["sex", "l1", "l2", "occupation", "age", "education"]
    for spk in sprecher:
        for feld in standardfelder:
            if feld not in sprecher[spk]:
                sprecher[spk][feld] = "unknown"

    return sprecher


### 4.4 Read timestamps

In [33]:
def parse_zeitstempel(zeile):
    """
    Extracts segment numbers and time information from a timestamp line.
    Returns: (start_time_str, end_time_str, list_segment_numbers)
    Example: '5 [00:05.0] 6 [00:06.0]' -> ('00:05.0', '00:06.0', [5, 6])
    """
    paare = re.findall(r"(\d+)\s*\[(\d+:\d+\.\d+)\]", zeile)
    if not paare:
        return None, None, []
    return paare[0][1], paare[-1][1], [int(p[0]) for p in paare]


### 4.5 Read turns

The core component: We read all turns line by line and build a dataset for each utterance.

An important special case: Lines beginning with `..` are continuations of the previous turn — the last segment of the predecessor is then adopted.


In [34]:
def parse_turns(turn_zeilen, bekannte_sprecher=None):
    """
  Reads all turns.
Returns a list of dicts — one utterance per entry.
Fields: turn_id, segment_start, segment_ende, segmente, sprecher, kanal, text

bekannte_sprecher: optional set of speaker IDs from the speaker table.
Required for files without [v] marking (alternative format).
    """
    bekannte_sprecher = bekannte_sprecher or set()
    saetze            = []
    aktueller_turn    = None
    aktueller_start   = None
    aktuelles_ende    = None
    aktuelle_segmente = []
    letzte_segmentnr  = None

    for z in turn_zeilen:
        kat = kategorisiere(z)

        if kat == "turn_nummer":
            aktueller_turn    = int(re.search(r"\d+", z).group())
            aktueller_start   = None
            aktuelles_ende    = None
            aktuelle_segmente = []

        elif kat == "zeitstempel":
            start, ende, segmente = parse_zeitstempel(z)
            if z.strip().startswith("..") and letzte_segmentnr is not None:
                segmente = [letzte_segmentnr] + segmente
            if aktueller_start is None:
                aktueller_start = start
            aktuelles_ende = ende
            aktuelle_segmente.extend(segmente)
            if segmente:
                letzte_segmentnr = segmente[-1]

        elif kat == "sprecher_verbal":
            # Standard-Format: "A1 [v] text"
            m = re.match(r"^(.+?)\s+\[v\]\s*(.*)", z.strip())
            if m:
                saetze.append({
                    "turn_id":       aktueller_turn,
                    "segment_start": aktueller_start,
                    "segment_ende":  aktuelles_ende,
                    "segmente":      aktuelle_segmente.copy(),
                    "sprecher":      m.group(1).strip(),
                    "kanal":         "v",
                    "text":          m.group(2).strip(),
                })

        elif kat == "nonverbal_solo":
            saetze.append({
                "turn_id":       aktueller_turn,
                "segment_start": aktueller_start,
                "segment_ende":  aktuelles_ende,
                "segmente":      aktuelle_segmente.copy(),
                "sprecher":      "[nv]",
                "kanal":         "nv",
                "text":          z.strip(),
            })

        elif kat == "sonstiges" and aktueller_turn is not None and bekannte_sprecher:
           # Alternative format without [v]: "PROF text..."
            # Check if first word is a known speaker ID
            teile = z.strip().split(None, 1)
            if teile and teile[0] in bekannte_sprecher:
                saetze.append({
                    "turn_id":       aktueller_turn,
                    "segment_start": aktueller_start,
                    "segment_ende":  aktuelles_ende,
                    "segmente":      aktuelle_segmente.copy(),
                    "sprecher":      teile[0],
                    "kanal":         "v",
                    "text":          teile[1].strip() if len(teile) > 1 else "",
                })

    return saetze


---
## Part 5 — Main Function: File → DataFrame

All parser functions are combined here into a single function. It takes a file path and returns a complete data table where each row is an utterance.

In [35]:
def datei_zu_dataframe(dateipfad):
    """
   Reads a GAT-2 transcription file (PDF or text file)
and returns a pandas DataFrame.

Columns: turn_id, segment_start, segment_ende, segmente,
         sprecher, kanal, text, datei,
         + speaker metadata (sex, l1, l2, beruf, ...)
    """
    # 1. Text lesen
    rohtext = rohtext_aus_datei(dateipfad)
    zeilen  = rohtext.splitlines()

    # 2. Abschnitte trennen
    idx_st = next((i for i, z in enumerate(zeilen)
                   if "speakertable" in z.lower()), None)
    idx_t1 = next((i for i, z in enumerate(zeilen)
                   if kategorisiere(z) == "turn_nummer"), None)

    if idx_st is None or idx_t1 is None:
        print(f"  WARNUNG: Struktur nicht erkannt — {Path(dateipfad).name}")
        return None

    header_zeilen       = zeilen[:idx_st]
    speakertable_zeilen = zeilen[idx_st:idx_t1]
    turn_zeilen         = zeilen[idx_t1:]

    # 3. Metadaten parsen
    header         = parse_header(header_zeilen)
    sprecher_beruf = parse_berufe(header)
    sprecher       = parse_speakertable(speakertable_zeilen, sprecher_beruf)

    # 4. Turns parsen
    saetze = parse_turns(turn_zeilen, bekannte_sprecher=set(sprecher.keys()))
    df = pd.DataFrame(saetze)
    df["datei"] = Path(dateipfad).stem

    # 5. Sprecher-Metadaten als Spalten ergänzen (wenn vorhanden)
    if sprecher and len(df) > 0 and "sprecher" in df.columns:
        meta = pd.DataFrame(sprecher).T
        meta.index.name = "sprecher"
        meta = meta.reset_index()
        df = df.merge(meta, on="sprecher", how="left")

    # Spaltenreihenfolge vereinheitlichen
    kernspalten = ["turn_id", "segment_start", "segment_ende", "segmente",
                   "sprecher", "kanal", "text", "datei",
                   "sex", "l1", "l2", "occupation", "age", "education"]
    vorhandene = [k for k in kernspalten if k in df.columns]
    df = df[vorhandene]

    return df


---
## Part 6 — Read All Files

Now we apply `datei_zu_dataframe()` to all transcription files. The result is a dict with one DataFrame per file.

In [36]:
dataframes = {}

print("Reading all transcription files...\n")
for pfad in alle_dateien:
    name = Path(pfad).stem
    df   = datei_zu_dataframe(pfad)

    if df is not None:
        dataframes[name] = df
        n_turns    = df["turn_id"].nunique()
        n_segmente = df["segmente"].explode().nunique()
        n_sprecher = df[df["kanal"] == "v"]["sprecher"].nunique()
        print(f"{name}")
        print(f"  Turns: {n_turns}  |  Segmente: {n_segmente}  "
              f"|  Sprecher: {n_sprecher}  |  Äußerungen: {len(df)}")
    print()

print(f"Total read: {len(dataframes)} files")


Reading all transcription files...

2012_Uberaba_Alemaes_Sueca1
  Turns: 1618  |  Segmente: 2741  |  Sprecher: 7  |  Äußerungen: 2927

2013_Muenster_Alemaes1-Parte1
  Turns: 1128  |  Segmente: 2769  |  Sprecher: 6  |  Äußerungen: 2512

2013_Muenster_Alemaes1-Parte2
  Turns: 491  |  Segmente: 1152  |  Sprecher: 6  |  Äußerungen: 1112

2014_BeloHorizonte_Alemao_Brasileiros_AulaPreposicao01_parte2
  Turns: 215  |  Segmente: 294  |  Sprecher: 8  |  Äußerungen: 393

2014_BeloHorizonte_Alemao_Brasileiros_AulaPreposicao01_parte3
  Turns: 239  |  Segmente: 524  |  Sprecher: 9  |  Äußerungen: 394

2014_Muenster_Alemaes2-Parte1
  Turns: 918  |  Segmente: 1915  |  Sprecher: 5  |  Äußerungen: 1859

2014_Muenster_Alemaes2-Parte2
  Turns: 211  |  Segmente: 519  |  Sprecher: 5  |  Äußerungen: 496

2014_Muenster_Brasileiros2-Parte1
  Turns: 761  |  Segmente: 1263  |  Sprecher: 6  |  Äußerungen: 1474

2014_Muenster_Brasileiros2-Parte2
  Turns: 839  |  Segmente: 1403  |  Sprecher: 5  |  Äußerungen: 1429

---
## Part 7 — Overview of the Loaded Data

Two exemplary evaluations show what the DataFrames contain.
Detailed linguistic analyses follow in the evaluation notebook.


In [37]:
# Example: First rows of a DataFrame
beispiel_name = list(dataframes.keys())[0]
df = dataframes[beispiel_name]

print(f"Beispieldatei: {beispiel_name}")
print(f"Spalten: {list(df.columns)}")
print(f"Zeilen: {len(df)}")
print()
df.head(5)


Beispieldatei: 2012_Uberaba_Alemaes_Sueca1
Spalten: ['turn_id', 'segment_start', 'segment_ende', 'segmente', 'sprecher', 'kanal', 'text', 'datei', 'sex', 'l1', 'l2', 'occupation', 'age', 'education']
Zeilen: 2927



,turn_id,segment_start,segment_ende,segmente,sprecher,kanal,text,datei,sex,l1,l2,occupation,age,education
0,1,00:00.0,00:07.7,"[0, 1, 2, 3, 4, 5]",A2,v,,2012_Uberaba_Alemaes_Sueca1,m,deu,eng; por,unknown,unknown,unknown
1,1,00:00.0,00:07.7,"[0, 1, 2, 3, 4, 5]",Operador 1,v,,2012_Uberaba_Alemaes_Sueca1,f,unknown,unknown,unknown,unknown,unknown
2,1,00:00.0,00:07.7,"[0, 1, 2, 3, 4, 5]",Operador 2,v,,2012_Uberaba_Alemaes_Sueca1,f,unknown,unknown,unknown,unknown,unknown
3,2,00:08.8,00:12.4,"[6, 7]",Operador 1,v,,2012_Uberaba_Alemaes_Sueca1,f,unknown,unknown,unknown,unknown,unknown
4,3,00:13.5,00:15.8,"[7, 8, 9, 11, 12]",A2,v,,2012_Uberaba_Alemaes_Sueca1,m,deu,eng; por,unknown,unknown,unknown


### 7.1 nonverbal_solo — Phases without Speaker

`[nv]` lines mark phases in which no one is speaking.
They do not appear in all transcripts — some projects
note pauses directly in the text of utterances instead,
e.g., as `(.)`, `(--)` or `(1.2)` within `[v]` lines.
The evaluation of pauses within utterances follows in the evaluation notebook.


In [38]:
pausen = df[df["kanal"] == "nv"].copy()

print(f"Pauses total: {len(pausen)}")
print(f"Pauses per turn (average): {len(pausen) / df['turn_id'].nunique():.2f}")
print()

# Extract pause duration from notation e.g. (1.2) or (2.5s)
pausen["dauer_sek"] = pd.to_numeric(
    pausen["text"].apply(
        lambda t: m.group(1) if (m := re.search(r"\((\d+\.?\d*)s?\)", str(t))) else None
    ),
    errors="coerce"
)

print("Pause durations (in seconds):")
if pausen["dauer_sek"].notna().any():
    print(pausen["dauer_sek"].describe().round(2))
else:
    print("  No parseable pause durations found.")
    if len(pausen) > 0:
        print(f"  Example texts: {pausen['text'].head(3).tolist()}")

Pauses total: 0
Pauses per turn (average): 0.00

Pause durations (in seconds):
  No parseable pause durations found.


### 7.2 Overall Overview of All Files


In [39]:
uebersicht = []
for name, df_i in dataframes.items():
    verbal_i = df_i[df_i["kanal"] == "v"]
    uebersicht.append({
        "datei":        name,
        "turns":        df_i["turn_id"].nunique(),
        "segmente":     df_i["segmente"].explode().nunique(),
        "aeusserungen": len(verbal_i),
        "pausen":       len(df_i[df_i["kanal"] == "nv"]),
        "sprecher":     verbal_i["sprecher"].nunique(),
    })

uebersicht_df = pd.DataFrame(uebersicht).set_index("datei")
print("Overview of all transcription files:\n")
print(uebersicht_df.to_string())


Overview of all transcription files:

                                                               turns  segmente  aeusserungen  pausen  sprecher
datei                                                                                                         
2012_Uberaba_Alemaes_Sueca1                                     1618      2741          2927       0         7
2013_Muenster_Alemaes1-Parte1                                   1128      2769          2512       0         6
2013_Muenster_Alemaes1-Parte2                                    491      1152          1112       0         6
2014_BeloHorizonte_Alemao_Brasileiros_AulaPreposicao01_parte2    215       294           393       0         8
2014_BeloHorizonte_Alemao_Brasileiros_AulaPreposicao01_parte3    239       524           394       0         9
2014_Muenster_Alemaes2-Parte1                                    918      1915          1859       0         5
2014_Muenster_Alemaes2-Parte2                                    211      

---
## Interim Conclusion — What Has Been Loaded So Far

The above parts 1–7 have loaded all PDF transcripts and shown initial evaluations.

In the next section (Part 6b), the EXMARaLDA CSV exports will be added
and everything will be merged into an overall table.


---
## Part 6b — Overall Table: Merging PDF + EXMARaLDA

All loaded DataFrames — from PDFs and from EXMARaLDA exports — are
merged here into a single table `df_gesamt`.

The column `quelle` indicates whether an entry originates from a PDF transcript
(`pdf`) or an EXMARaLDA export (`exmaralda`).

Note: If a recording exists both as PDF and as EXMARaLDA CSV,
it will appear with different `datei` names.
Similar names (e.g., `2016_..._Assembleia2` and `assembleia2`) can
be harmonized later by renaming in the `datei` field.


In [40]:
# Load EXMARaLDA CSV exports
exmaralda_frames = {}
print("Loading EXMARaLDA CSV exports...\n")
for pfad in exmaralda_dateien:
    name  = Path(pfad).stem
    meta  = EXMARALDA_META.get(name)
    df_ex = exmaralda_zu_dataframe(pfad, sprecher_meta=meta)

    if df_ex is not None and len(df_ex) > 0:
        exmaralda_frames[name] = df_ex
        n_turns    = df_ex["turn_id"].nunique()
        n_sprecher = df_ex[df_ex["kanal"] == "v"]["sprecher"].nunique()
        print(f"{name}")
        print(f"  Turns: {n_turns}  |  Speakers: {n_sprecher}  |  Utterances: {len(df_ex)}")
    else:
        print(f"{name}  ⚠ skipped (empty or unreadable)")
    print()

print(f"Total loaded: {len(exmaralda_frames)} EXMARaLDA file(s)")

Loading EXMARaLDA CSV exports...

assembleia1
  Turns: 1  |  Speakers: 6  |  Utterances: 2373

assembleia2
  Turns: 7  |  Speakers: 6  |  Utterances: 3737

assembleia_2
  Turns: 7  |  Speakers: 6  |  Utterances: 3737

icmi_speaker_metadata  ⚠ skipped (empty or unreadable)

modalpartikeln
  Turns: 1  |  Speakers: 8  |  Utterances: 1758

Total loaded: 4 EXMARaLDA file(s)


In [41]:
# Merge all DataFrames into one overall table

# Mark source
pdf_liste = []
for name, df_i in dataframes.items():
    df_copy = df_i.copy()
    df_copy["quelle"] = "pdf"
    pdf_liste.append(df_copy)

ex_liste = []
for name, df_i in exmaralda_frames.items():
    df_copy = df_i.copy()
    df_copy["quelle"] = "exmaralda"
    ex_liste.append(df_copy)

alle_frames = pdf_liste + ex_liste

if alle_frames:
    df_gesamt = pd.concat(alle_frames, ignore_index=True)
    print("df_gesamt created:")
    print(f"  Total rows        : {len(df_gesamt):>7,}")
    print(f"  From PDFs         : {sum(len(d) for d in pdf_liste):>7,}  ({len(pdf_liste)} files)")
    print(f"  From EXMARaLDA    : {sum(len(d) for d in ex_liste):>7,}  ({len(ex_liste)} files)")
    print(f"  Columns           : {list(df_gesamt.columns)}")
    print(f"\nPreview (5 rows):")
    print(df_gesamt[["datei","quelle","sprecher","kanal","text"]].head())
else:
    print("No DataFrames found — df_gesamt could not be created.")


df_gesamt created:
  Total rows        :  32,377
  From PDFs         :  20,772  (13 files)
  From EXMARaLDA    :  11,605  (4 files)
  Columns           : ['turn_id', 'segment_start', 'segment_ende', 'segmente', 'sprecher', 'kanal', 'text', 'datei', 'sex', 'l1', 'l2', 'occupation', 'age', 'education', 'quelle']

Preview (5 rows):
                         datei quelle    sprecher kanal text
0  2012_Uberaba_Alemaes_Sueca1    pdf          A2     v     
1  2012_Uberaba_Alemaes_Sueca1    pdf  Operador 1     v     
2  2012_Uberaba_Alemaes_Sueca1    pdf  Operador 2     v     
3  2012_Uberaba_Alemaes_Sueca1    pdf  Operador 1     v     
4  2012_Uberaba_Alemaes_Sueca1    pdf          A2     v     


In [42]:
# Overview of the overall table by file

uebersicht_gesamt = []
for datei_name, gruppe in df_gesamt.groupby("datei", sort=True):
    verbal   = gruppe[gruppe["kanal"] == "v"]
    uebersicht_gesamt.append({
        "datei":        datei_name,
        "quelle":       gruppe["quelle"].iloc[0],
        "turns":        gruppe["turn_id"].nunique(),
        "aeusserungen": len(verbal),
        "pausen":       len(gruppe[gruppe["kanal"] == "nv"]),
        "sprecher":     verbal["sprecher"].nunique(),
    })

uebersicht_gesamt_df = pd.DataFrame(uebersicht_gesamt).set_index("datei")
print("Overview of all sources:\n")
print(uebersicht_gesamt_df.to_string())


Overview of all sources:

                                                                  quelle  turns  aeusserungen  pausen  sprecher
datei                                                                                                          
2012_Uberaba_Alemaes_Sueca1                                          pdf   1618          2927       0         7
2013_Muenster_Alemaes1-Parte1                                        pdf   1128          2512       0         6
2013_Muenster_Alemaes1-Parte2                                        pdf    491          1112       0         6
2014_BeloHorizonte_Alemao_Brasileiros_AulaPreposicao01_parte2        pdf    215           393       0         8
2014_BeloHorizonte_Alemao_Brasileiros_AulaPreposicao01_parte3        pdf    239           394       0         9
2014_Muenster_Alemaes2-Parte1                                        pdf    918          1859       0         5
2014_Muenster_Alemaes2-Parte2                                        pdf    21

In [43]:
# Save overall table as CSV
import os
os.makedirs("csv_export", exist_ok=True)

ausgabepfad = "csv_export/korpus_gesamt.csv"
df_gesamt.to_csv(ausgabepfad, index=False, encoding="utf-8-sig")
print(f"Overall table saved: {ausgabepfad}")
print(f"  {len(df_gesamt):,} rows, {len(df_gesamt.columns)} columns")

Overall table saved: csv_export/korpus_gesamt.csv
  32,377 rows, 15 columns


---
## Part 8 — Export as CSV

The finished DataFrames are saved as CSV files.
CSV (Comma-Separated Values) is a simple table format
that can be read by Excel, R, spaCy, and all other analysis programs.

Each file is saved under its original name,
with the extension `_auswertung.csv`.
The encoding `utf-8-sig` ensures that umlauts and special characters
are displayed correctly even in Excel.


In [44]:
import os

os.makedirs("csv_export", exist_ok=True)

for name, df_i in dataframes.items():
    ausgabepfad = f"csv_export/{name}_auswertung.csv"
    df_i.to_csv(ausgabepfad, index=False, encoding="utf-8-sig")
    print(f"Saved: {ausgabepfad}  ({len(df_i)} rows, {len(df_i.columns)} columns)")

print(f"\nAll {len(dataframes)} files exported.")


Saved: csv_export/2012_Uberaba_Alemaes_Sueca1_auswertung.csv  (2927 rows, 14 columns)
Saved: csv_export/2013_Muenster_Alemaes1-Parte1_auswertung.csv  (2512 rows, 14 columns)
Saved: csv_export/2013_Muenster_Alemaes1-Parte2_auswertung.csv  (1112 rows, 14 columns)
Saved: csv_export/2014_BeloHorizonte_Alemao_Brasileiros_AulaPreposicao01_parte2_auswertung.csv  (393 rows, 14 columns)
Saved: csv_export/2014_BeloHorizonte_Alemao_Brasileiros_AulaPreposicao01_parte3_auswertung.csv  (394 rows, 14 columns)
Saved: csv_export/2014_Muenster_Alemaes2-Parte1_auswertung.csv  (1859 rows, 14 columns)
Saved: csv_export/2014_Muenster_Alemaes2-Parte2_auswertung.csv  (496 rows, 14 columns)
Saved: csv_export/2014_Muenster_Brasileiros2-Parte1_auswertung.csv  (1474 rows, 14 columns)
Saved: csv_export/2014_Muenster_Brasileiros2-Parte2_auswertung.csv  (1429 rows, 14 columns)
Saved: csv_export/2014_Muenster_Brasileiros2-Parte3_auswertung.csv  (1247 rows, 14 columns)
Saved: csv_export/2015_BeloHorizonte_Alemas_Brasi

---
## Outlook — Possible Extensions

The exported CSV files can serve as a starting point for further
analyses — in Python itself or in other programs.

**In Python:**
- Visualizations with `matplotlib` or `seaborn`
  (e.g., bar charts of speaker shares, time series)
- Collocation analyses and lemmatization with `spaCy`
- Statistical tests with `scipy` or `statsmodels`

**In other programs:**
- **Excel / LibreOffice Calc**: Open CSV directly,
  create pivot tables and charts
- **R**: `read.csv()` + `tidyverse` for statistical analyses
- **ELAN**: Reimport of annotations possible
